# Romanization Pipeline for Indic MT Evaluation

**Paper:** *Lost in Transliteration: Orthographic Sensitivity in Neural MT Evaluation*  
**Authors:** John Salvin, Swapnil Hingmire · IIT Palakkad · 2026

## What This Notebook Does

This notebook implements the **script-normalization (romanization) pipeline** used in the paper.  
It converts Indic-script translations and references into ITRANS romanized form using the
`indic-transliteration` library, then merges those transliterations back into the MQM evaluation
dataset and computes neural MT metric scores (BERTScore, BLEURT, COMET) on **both**
the original Indic-script text and the romanized text.

## Pipeline Steps

| Step | Description |
|------|-------------|
| Setup | Import libraries, install dependencies |
| Load Data | Load 5-language IndicMT MQM dataset |
| Romanization Class | Define `RomanizationPipeline` (ITRANS via `indic-transliteration`) |
| Romanize | Apply romanization to all 5 languages (1,400 samples each) |
| Quality Check | Manual inspection of romanization output |
| Merge Back | Add romanized columns to original dataset |
| Post-processing | Lowercase + clean whitespace/zero-width chars |
| Export CSVs (MATEO format) | Save per-language CSV/TSV files for MATEO upload |
| Merge Metrics | Join BERTScore / BLEURT / COMET scores into dataset |
| Compute Neural Metrics Locally | Run BERTScore, BLEURT, COMET on CPU or GPU |

## Languages Covered

| Language | Script | ITRANS System |
|----------|--------|---------------|
| Hindi | Devanagari | `sanscript.DEVANAGARI` |
| Marathi | Devanagari | `sanscript.DEVANAGARI` |
| Gujarati | Gujarati | `sanscript.GUJARATI` |
| Tamil | Tamil | `sanscript.TAMIL` |
| Malayalam | Malayalam | `sanscript.MALAYALAM` |

## Data

Uses the **IndicMT Eval** dataset (Sindhujan et al., 2023).  
Download from: https://github.com/AI4Bharat/IndicMT-Eval  
Place the file at: `data/Indic_MT_MQM_data.xlsx`

## Dependencies

```
pip install -r requirements.txt
```

See `requirements.txt` in the root of the repository.

---
> ⚠️ **Reproducibility Note:** The local neural metric scoring section (last section)
> can run on CPU or GPU. BERTScore, BLEURT, and COMET were originally computed via
> [MATEO](https://mateo.ivdnt.org/) — the export section below generates the files
> in the exact format required for upload.


## Setup

Import all required libraries. Run the commented `pip install` lines once if you haven't
installed the dependencies yet.


In [ ]:
# ─── Standard library imports ───────────────────────────────────────────────
import os
import re
import warnings
from pathlib import Path

# ─── Third-party imports ─────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from tqdm import tqdm

warnings.filterwarnings('ignore')

# ─── Install romanization library (run once) ─────────────────────────────────
# !pip install indic-transliteration

print("✓ Libraries loaded successfully")


## Load Data

Loads the IndicMT MQM dataset from an Excel file with one sheet per language.
Each sheet contains **1,400 sentences** with `Source`, `Translation`, and `Reference` columns.

> 📂 **Expected file path:** `data/Indic_MT_MQM_data.xlsx`  
> If your file is stored elsewhere, update `DATA_PATH` below.


In [ ]:
# ─── Configuration ───────────────────────────────────────────────────────────
DATA_PATH = Path("data/Indic_MT_MQM_data.xlsx")

LANGUAGES = {
    'hi': ('Hindi',     'Devanagari'),
    'mr': ('Marathi',   'Devanagari'),
    'gu': ('Gujarati',  'Gujarati'),
    'ta': ('Tamil',     'Tamil'),
    'ml': ('Malayalam', 'Malayalam'),
}

# ─── Load all language sheets ─────────────────────────────────────────────────
dataframes = {}
for lang_code, (sheet_name, script) in LANGUAGES.items():
    try:
        df = pd.read_excel(DATA_PATH, sheet_name=sheet_name)
        dataframes[lang_code] = df
        print(f"✓ {sheet_name:10s} ({script:11s}): {len(df):,} samples")
    except Exception as e:
        print(f"✗ Could not load {sheet_name}: {e}")

print(f"\nTotal languages loaded: {len(dataframes)}/5")


## Romanization Pipeline

Defines the `RomanizationPipeline` class that wraps the `indic-transliteration` library.

**Transliteration scheme used:** [ITRANS](https://www.aczoom.com/itrans/) — a widely-used
ASCII representation of Indic phonemes. Chosen because it is deterministic, reversible,
and supported for all five target scripts by the `indic-transliteration` library
(Murthy & Kumar, 2020).

**Reference:**
> Arun Murthy and Shrirang Kulkarni. *indic-transliteration: Script Conversion for Indic Languages.*  
> https://github.com/indic-transliteration/indic_transliteration (2020)


In [ ]:
from indic_transliteration import sanscript


class RomanizationPipeline:
    """
    Converts Indic-script text to ITRANS romanization.

    Supports Hindi and Marathi (Devanagari), Gujarati, Tamil, and Malayalam.
    Uses the `indic-transliteration` library's `sanscript.transliterate()`
    function under the hood.

    Usage:
        romanizer = RomanizationPipeline()
        roman = romanizer.romanize_text("नमस्ते", "hi")  # → 'namaste'
    """

    SCRIPT_MAP = {
        'hi': sanscript.DEVANAGARI,
        'mr': sanscript.DEVANAGARI,
        'gu': sanscript.GUJARATI,
        'ta': sanscript.TAMIL,
        'ml': sanscript.MALAYALAM,
    }

    SCRIPT_NAMES = {
        'hi': 'Devanagari', 'mr': 'Devanagari',
        'gu': 'Gujarati',   'ta': 'Tamil',   'ml': 'Malayalam',
    }

    def romanize_text(self, text: str, lang_code: str) -> str:
        """
        Romanize a single string from an Indic script to ITRANS.

        Args:
            text      : Input string in the native Indic script.
            lang_code : ISO 639-1 code (e.g. 'hi', 'ta').

        Returns:
            Romanized string in ITRANS encoding, or the original string
            if the language is unsupported or an error occurs.
        """
        if pd.isna(text) or str(text).strip() == '':
            return ''
        script = self.SCRIPT_MAP.get(lang_code)
        if script is None:
            return str(text)
        try:
            result = sanscript.transliterate(str(text), script, sanscript.ITRANS)
            return result if result else str(text)
        except Exception as e:
            print(f"[WARN] Romanization error for '{str(text)[:40]}': {e}")
            return str(text)

    def romanize_dataframe(self, df: pd.DataFrame, lang_code: str) -> pd.DataFrame:
        """
        Apply romanization to the 'Reference' and 'Translation' columns of a DataFrame.

        Adds four new columns:
            - Reference_romanized        : ITRANS romanization of the reference.
            - Translation_romanized      : ITRANS romanization of the MT output.
            - Language                   : Uppercase language code (e.g. 'HI').
            - Script_Original            : Name of the source script.

        Args:
            df        : DataFrame with 'Reference' and 'Translation' columns.
            lang_code : ISO 639-1 code for the language.

        Returns:
            A new DataFrame with the added columns.
        """
        df = df.copy()
        tqdm.pandas(desc=f"Romanizing {lang_code.upper()}")
        df['Reference_romanized']   = df['Reference'].progress_apply(
            lambda x: self.romanize_text(x, lang_code))
        df['Translation_romanized'] = df['Translation'].progress_apply(
            lambda x: self.romanize_text(x, lang_code))
        df['Language']        = lang_code.upper()
        df['Script_Original'] = self.SCRIPT_NAMES.get(lang_code, 'Unknown')
        return df


romanizer = RomanizationPipeline()
print("✓ RomanizationPipeline ready")


## Apply Romanization

Runs the romanization pipeline on all five languages.
Each language has **1,400 sentence pairs** (source, MT translation, human reference).
Processing is fast — all five languages complete in under 5 seconds on CPU.


In [ ]:
romanized = {}

for lang_code, (sheet_name, _) in LANGUAGES.items():
    if lang_code not in dataframes:
        print(f"[SKIP] {sheet_name} not loaded — skipping romanization.")
        continue
    romanized[lang_code] = romanizer.romanize_dataframe(dataframes[lang_code], lang_code)
    print(f"✓ {sheet_name}: {len(romanized[lang_code]):,} samples romanized")

print("\nRomanization complete.")


## Quality Assurance — Manual Inspection

Prints 5 random sentence pairs per language so you can visually verify
that the romanization looks correct. Uses `random_state=42` for reproducibility.

**What to look for:**
- Are vowel markers (mātrā) preserved correctly?
- Are geminate consonants doubled (e.g. `tt`, `kk`)?
- Does Tamil retroflex vs. dental distinction appear?


In [ ]:
def qa_inspect(df: pd.DataFrame, lang_name: str, n: int = 5) -> None:
    """Print n random (original, romanized) translation pairs for manual inspection."""
    print(f"\n{'='*65}")
    print(f"  {lang_name} — {n} Random Sample Pairs")
    print(f"{'='*65}")
    for _, row in df.sample(n, random_state=42).iterrows():
        print(f"  Original  : {str(row['Translation'])[:80]}")
        print(f"  Romanized : {str(row['Translation_romanized'])[:80]}")
        print(f"  {'-'*60}")


for lang_code, (sheet_name, _) in LANGUAGES.items():
    if lang_code in romanized:
        qa_inspect(romanized[lang_code], sheet_name)


## Merge Romanized Columns into Original Dataset

Adds `Reference_Transliteration` and `Translation_Transliteration` columns
to the original Excel file. A **row-alignment check** is run first to ensure
the romanized data matches the original rows (verifies the first reference sentence).

The output file path is set by `OUTPUT_PATH` below.


In [ ]:
OUTPUT_PATH = Path("data/Indic_MT_MQM_data.xlsx")

print("Verifying row alignment between original and romanized data:")
for lang_code, (sheet_name, _) in LANGUAGES.items():
    if lang_code not in dataframes or lang_code not in romanized:
        continue
    orig_ref  = dataframes[lang_code]['Reference'].iloc[0]
    roman_ref = romanized[lang_code]['Reference'].iloc[0]
    status = "✓" if (orig_ref == roman_ref) else "✗ MISMATCH"
    print(f"  {status} {sheet_name}")

updated = {}
for lang_code, (sheet_name, _) in LANGUAGES.items():
    if lang_code not in dataframes or lang_code not in romanized:
        continue
    df = dataframes[lang_code].copy()
    df['Reference_Transliteration']   = romanized[lang_code]['Reference_romanized'].values
    df['Translation_Transliteration'] = romanized[lang_code]['Translation_romanized'].values
    updated[sheet_name] = df

with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    for sheet_name, df in updated.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"\n✓ Saved with transliteration columns to: {OUTPUT_PATH}")
print("  Added columns: Reference_Transliteration, Translation_Transliteration")


## Post-processing — Clean Transliterations

Applies light text normalization to the romanized columns to produce
`*_Transliteration_clean` variants. These clean versions are what get
passed to the MT metric models.

**Normalization steps:**
1. Remove Unicode zero-width joiners (`\u200c`, `\u200d`) common in Indic text
2. Collapse multiple whitespace characters to a single space
3. Add spaces around punctuation (`.`, `,`, `;`, `:`, `!`, `?`) for consistent tokenization
4. Lowercase everything


In [ ]:
def clean_transliteration(text: str) -> str:
    """
    Normalize a romanized (ITRANS) string for downstream metric evaluation.

    Steps:
        1. Strip zero-width Unicode characters (\u200c, \u200d).
        2. Collapse multiple whitespace into a single space.
        3. Pad punctuation with spaces for consistent tokenization.
        4. Lowercase.
    """
    if pd.isna(text):
        return ''
    s = str(text)
    s = s.replace('\u200c', '').replace('\u200d', '')
    s = re.sub(r'\s+', ' ', s).strip()
    s = re.sub(r'([.,;:!?])', r' \1 ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s.lower()


def add_clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add *_clean variants of the transliteration columns to a DataFrame."""
    df = df.copy()
    if 'Reference_Transliteration' in df.columns:
        df['Reference_Transliteration_clean']   = df['Reference_Transliteration'].apply(clean_transliteration)
    if 'Translation_Transliteration' in df.columns:
        df['Translation_Transliteration_clean'] = df['Translation_Transliteration'].apply(clean_transliteration)
    return df


cleaned_dfs = {}
for lang_code, (sheet_name, _) in LANGUAGES.items():
    df = pd.read_excel(OUTPUT_PATH, sheet_name=sheet_name)
    cleaned_dfs[sheet_name] = add_clean_columns(df)

with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    for sheet_name, df in cleaned_dfs.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print("✓ Clean transliteration columns added:")
print("   Reference_Transliteration_clean")
print("   Translation_Transliteration_clean")
print(f"  Saved to: {OUTPUT_PATH}")


## Export Files for MATEO (Per-Language CSV & TSV)

[MATEO (MAchine Translation Evaluation Online)](https://mateo.ivdnt.org/) is a free web tool
developed at Ghent University that computes BERTScore, BLEURT, and COMET without any local
installation. It requires separate files for **source**, **reference**, and **translation** —
one sentence per line, no header, UTF-8 encoded.

This section exports those files for each language into `data/mateo/<language>/`.

### How to use MATEO

1. Go to [https://mateo.ivdnt.org/Evaluate](https://mateo.ivdnt.org/Evaluate)
2. Upload the `source.csv`, `reference_transliteration_clean.csv`, and
   `translation_transliteration_clean.csv` files for one language at a time
3. Select metrics: BERTScore, BLEURT, COMET
4. Click **Evaluate** and download the results CSV
5. Repeat for each language

> ⚠️ Upload each language separately — MATEO evaluates one language pair at a time.
> The generated metric scores can then be fed into the Merge Metrics section below.

### Citing MATEO

If you use MATEO in your work, please cite:

```bibtex
@inproceedings{vanroy-etal-2023-mateo,
    title     = "{MATEO}: {MA}chine {T}ranslation {E}valuation {O}nline",
    author    = "Vanroy, Bram and Tezcan, Arda and Macken, Lieve",
    booktitle = "Proceedings of the 24th Annual Conference of the European Association for Machine Translation",
    year      = "2023",
    url       = "https://aclanthology.org/2023.eamt-1.52",
    pages     = "499--500",
}
```


In [ ]:
MATEO_BASE = Path("data/mateo")

def export_mateo_files(sheet_name: str) -> None:
    """
    Export source, reference, and translation files for MATEO for one language.

    Writes both CSV and TSV versions (no header, one sentence per line, UTF-8).
    Files are saved to data/mateo/<sheet_name>/

    Args:
        sheet_name : Name of the language sheet (e.g. 'Hindi').
    """
    out_dir = MATEO_BASE / sheet_name
    out_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_excel(OUTPUT_PATH, sheet_name=sheet_name)

    ref_col = next(
        (c for c in ['Reference_Transliteration_clean', 'Reference_Transliteration']
         if c in df.columns), None
    )
    tr_col = next(
        (c for c in ['Translation_Transliteration_clean', 'Translation_Transliteration']
         if c in df.columns), None
    )

    files_to_write = {
        'source':    df['Source'],
        'reference': df[ref_col] if ref_col else None,
        'translation': df[tr_col] if tr_col else None,
    }

    for fname, series in files_to_write.items():
        if series is None:
            print(f"  [WARN] {sheet_name}: missing column for '{fname}' — skipped.")
            continue
        # CSV (no header, UTF-8)
        series.to_csv(out_dir / f"{fname}.csv", index=False, header=False, encoding='utf-8')
        # TSV (no header, UTF-8)
        series.to_csv(out_dir / f"{fname}.tsv", index=False, header=False,
                      sep='\t', encoding='utf-8')

    print(f"✓ {sheet_name:10s} → {out_dir}/")
    print(f"   source.csv / source.tsv")
    print(f"   reference.csv / reference.tsv")
    print(f"   translation.csv / translation.tsv")


for _, (sheet_name, _) in LANGUAGES.items():
    export_mateo_files(sheet_name)

print(f"\n✓ MATEO files ready in: {MATEO_BASE}/")
print("  Upload each language folder separately to https://mateo.ivdnt.org/Evaluate")


## Merge Pre-computed Neural Metric Scores

Merges BERTScore, BLEURT, and COMET scores (computed via MATEO or locally) into the
main dataset. Scores are joined by matching the `Source` sentence text; a
**positional fallback** is used for any rows where the text match fails.

**Input file:** `data/Neural_metrics.xlsx`  
Sheets: `bertscore`, `bleurt`, `comet`  
Required columns in each sheet: `src`, `translation_transliteration_clean_score`

**References:**
- BERTScore: Zhang et al. (2020) — https://arxiv.org/abs/1904.09675
- BLEURT: Sellam et al. (2020) — https://arxiv.org/abs/2004.04696
- COMET: Rei et al. (2020) — https://arxiv.org/abs/2009.09025


In [ ]:
METRICS_FILE = Path("data/Neural_metrics.xlsx")


def normalize_key(text) -> str | None:
    """Normalize a source sentence for dictionary key lookup."""
    if pd.isna(text):
        return None
    s = str(text).strip().strip("\"'")
    return ' '.join(s.split()).lower()


def load_metric_scores(sheet_name: str) -> tuple[dict, pd.Series]:
    """
    Load metric scores from one sheet of the metrics Excel file.

    Returns:
        A tuple of:
            - mapping : dict mapping normalized source sentence → score
            - series  : pd.Series of scores in row order (for positional fallback)
    """
    df = pd.read_excel(METRICS_FILE, sheet_name=sheet_name)
    df = df[df['src'].notna()]
    df = df[df['src'].astype(str).str.lower() != 'source']
    df['key'] = df['src'].apply(normalize_key)
    mapping = df.set_index('key')['translation_transliteration_clean_score'].to_dict()
    series  = df['translation_transliteration_clean_score'].reset_index(drop=True)
    return mapping, series


bert_map,   bert_series   = load_metric_scores('bertscore')
bleurt_map, bleurt_series = load_metric_scores('bleurt')
comet_map,  comet_series  = load_metric_scores('comet')
print("✓ Metric score sheets loaded")

merged_dfs = {}
for lang_code, (sheet_name, _) in LANGUAGES.items():
    df = pd.read_excel(OUTPUT_PATH, sheet_name=sheet_name)
    df['__key'] = df['Source'].apply(normalize_key)

    df['bertscore'] = df['__key'].map(bert_map)
    df['bleurt']    = df['__key'].map(bleurt_map)
    df['comet']     = df['__key'].map(comet_map)

    n = len(df)
    for col, series in [('bertscore', bert_series), ('bleurt', bleurt_series),
                        ('comet', comet_series)]:
        mask = df[col].isna()
        if mask.any() and len(series) >= n:
            df.loc[mask, col] = series.iloc[:n].values[mask]

    missing = df[['bertscore', 'bleurt', 'comet']].isna().any(axis=1).sum()
    if missing > 0:
        print(f"[WARN] {sheet_name}: {missing} rows still missing metric scores after fallback")

    df = df.drop(columns='__key')
    merged_dfs[sheet_name] = df

with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    for sheet_name, df in merged_dfs.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"\n✓ BERTScore, BLEURT, COMET scores merged into: {OUTPUT_PATH}")


## Compute Neural Metrics Locally (CPU or GPU)

If you cannot use MATEO (e.g., large dataset, private data, or need batch processing),
you can compute BERTScore, BLEURT, and COMET directly on your machine.

### Installation

Install in this exact order to avoid version conflicts:

```bash
pip install "transformers==4.40.2"
pip install "protobuf==4.25.3"
pip install "bert-score==0.3.12" --force-reinstall --no-deps
pip install git+https://github.com/google-research/bleurt.git@cebe7e6
pip install "unbabel-comet==2.2.6"
```

### CPU vs GPU

Change `DEVICE` in the configuration block below:

| Setting | Value | Speed |
|---------|-------|-------|
| CPU | `DEVICE = "cpu"` | ~10–30 min for 7,000 sentences |
| GPU (CUDA) | `DEVICE = "cuda"` | ~2–5 min for 7,000 sentences |

### Models Used

| Metric | Model | Size | Notes |
|--------|-------|------|-------|
| BERTScore | `microsoft/mdeberta-v3-base` | ~900 MB | Multilingual; explicit `model_type=` avoids AutoModel bug in transformers ≥ 4.41 |
| BLEURT | `BLEURT-20` checkpoint | ~1.2 GB | Auto-downloaded on first run. Manual: [BLEURT-20.zip](https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip) |
| COMET | `Unbabel/wmt22-comet-da` | ~1.8 GB | Standard DA model; requires `source`, `target`, `refA` columns |

> ❌ **COMET-XL is not used in this work.** All COMET scores use `Unbabel/wmt22-comet-da`.

The script saves an intermediate checkpoint CSV after each metric so you don't lose
progress if a later step fails.


In [ ]:
# ─── protobuf fix: must be set BEFORE any other imports ──────────────────────
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import torch

# ─── Device configuration ─────────────────────────────────────────────────────
# Change DEVICE to "cuda" if you have a GPU available.
DEVICE = "cpu"   # Options: "cpu" | "cuda"

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"CUDA available : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU : {torch.cuda.get_device_name(0)}")


In [ ]:
# ─── Load the cleaned dataset ─────────────────────────────────────────────────
# This section operates on one language at a time to keep memory usage low.
# Change LANG_CODE to process a different language.

LANG_CODE  = 'hi'   # Options: 'hi', 'mr', 'gu', 'ta', 'ml'
SHEET_NAME = LANGUAGES[LANG_CODE][0]

df_local = pd.read_excel(OUTPUT_PATH, sheet_name=SHEET_NAME)
print(f"Loaded  {SHEET_NAME}: {len(df_local):,} rows")
print(f"Columns: {list(df_local.columns)}")

# Use romanized text as the MT output and reference for neural metrics
hyps = df_local['Translation_Transliteration_clean'].astype(str).tolist()
refs = df_local['Reference_Transliteration_clean'].astype(str).tolist()
srcs = df_local['Source'].astype(str).tolist()


In [ ]:
# ─── BERTScore ────────────────────────────────────────────────────────────────
# Version 0.3.12 | Model: microsoft/mdeberta-v3-base
# model_type= is specified explicitly to bypass AutoModel resolution bug
# in transformers >= 4.41.

from bert_score import score as bertscore_fn

BERTSCORE_MODEL = "microsoft/mdeberta-v3-base"

print(f"\n{'='*60}")
print(f"[BERTScore] {SHEET_NAME}  —  {len(df_local):,} rows")
print(f"  model  : {BERTSCORE_MODEL}")
print(f"  device : {DEVICE}")
print(f"{'='*60}")

P, R, F1 = bertscore_fn(
    cands=hyps,
    refs=refs,
    model_type=BERTSCORE_MODEL,
    verbose=True,
    batch_size=64,
    device=DEVICE,
)

df_local['BERTScore_P']  = [round(v, 4) for v in P.tolist()]
df_local['BERTScore_R']  = [round(v, 4) for v in R.tolist()]
df_local['BERTScore_F1'] = [round(v, 4) for v in F1.tolist()]

print(f"  Mean P={df_local['BERTScore_P'].mean():.4f}  "
      f"R={df_local['BERTScore_R'].mean():.4f}  "
      f"F1={df_local['BERTScore_F1'].mean():.4f}")

# Save checkpoint
checkpoint_path = Path(f"results/{SHEET_NAME.lower()}_metrics_checkpoint.csv")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
df_local.to_csv(checkpoint_path, index=False)
print(f"\n[Checkpoint] Saved after BERTScore → {checkpoint_path}")


In [ ]:
# ─── BLEURT ───────────────────────────────────────────────────────────────────
# Commit cebe7e6 | Checkpoint: BLEURT-20 (~1.2 GB, auto-downloaded on first run)
# Manual download: https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip

from bleurt import score as bleurt_score

BLEURT_CHECKPOINT = "BLEURT-20"

print(f"\n{'='*60}")
print(f"[BLEURT] {SHEET_NAME}  —  {len(df_local):,} rows")
print(f"  checkpoint : {BLEURT_CHECKPOINT}")
print(f"{'='*60}")

scorer = bleurt_score.BleurtScorer(BLEURT_CHECKPOINT)
scores = scorer.score(references=refs, candidates=hyps, batch_size=64)

df_local['BLEURT'] = [round(s, 4) for s in scores]
print(f"  Mean BLEURT = {df_local['BLEURT'].mean():.4f}")

df_local.to_csv(checkpoint_path, index=False)
print(f"\n[Checkpoint] Saved after BLEURT → {checkpoint_path}")


In [ ]:
# ─── COMET ────────────────────────────────────────────────────────────────────
# Version 2.2.6 | Model: Unbabel/wmt22-comet-da (~1.8 GB, auto-downloaded)
#
# ❌ COMET-XL is NOT used in this work.
#    All COMET scores use Unbabel/wmt22-comet-da (standard DA model).
#
# COMET requires source + translation + reference for each sentence.

from comet import download_model, load_from_checkpoint

COMET_MODEL = "Unbabel/wmt22-comet-da"

print(f"\n{'='*60}")
print(f"[COMET] {SHEET_NAME}  —  {len(df_local):,} rows")
print(f"  model  : {COMET_MODEL}")
print(f"  device : {DEVICE}  (gpus=0 means CPU)")
print(f"{'='*60}")

comet_model_path = download_model(COMET_MODEL)
comet_model      = load_from_checkpoint(comet_model_path)

data = [
    {"src": s, "mt": h, "ref": r}
    for s, h, r in zip(srcs, hyps, refs)
]

output = comet_model.predict(
    data,
    batch_size=64,
    gpus=0 if DEVICE == "cpu" else 1,
)

df_local['COMET'] = [round(s, 4) for s in output.scores]
print(f"  Mean COMET = {df_local['COMET'].mean():.4f}")

df_local.to_csv(checkpoint_path, index=False)
print(f"\n[Checkpoint] Final save → {checkpoint_path}")

# ─── Summary ──────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"DONE — All metrics computed for {SHEET_NAME}")
print(f"{'='*60}")
for col in ['BERTScore_P', 'BERTScore_R', 'BERTScore_F1', 'BLEURT', 'COMET']:
    if col in df_local.columns:
        print(f"  {col:18s} : {df_local[col].mean():.4f}")
